# 01. Validate the SQL return views in Python

**Purpose.** The research specification requires every important calculation to be checked independently. The SQL views `analytics.return_daily` and `analytics.excess_return_daily` produce the daily returns, the risk-free rate, the excess returns and the regime labels. This notebook recomputes them in pandas from the raw prices and compares the two.

**Metrics covered.** M1 simple daily return, M2 cumulative return, M5 daily risk-free rate and excess return (see `metric_definitions.md`).

**How to run.** Select the project `.venv` as the kernel, then use *Restart* and *Run All*. The notebook asks for the PostgreSQL user and password at the first code cell. It only reads from the database. Do not type the password into a cell.

**Pass criteria.** Every check below prints PASS. A FAIL means the SQL and Python calculations disagree and the analysis must not proceed until the cause is found.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
import db_access

pd.set_option("display.width", 160)
print("project root:", ROOT)

project root: d:\Data Analyst\Portfolio


## 1. Load the data

Two reads, both `SELECT` only:
- `core.price_daily` gives the prices that the Python side uses to recompute returns on its own.
- `analytics.excess_return_daily` gives what SQL computed.

The connection is closed straight after loading, so nothing stays open while the checks run.

In [2]:
conn = db_access.connect()
try:
    prices = db_access.load_prices(conn)
    sql = db_access.load_excess_returns(conn)
finally:
    conn.close()

print("prices rows:", len(prices), "| SQL view rows:", len(sql))
prices.head()

prices rows: 5536 | SQL view rows: 5524


,ticker,trade_date,close,adj_close
0,BBCA,2020-12-28,6780.0,5688.759277
1,BBCA,2020-12-29,6765.0,5676.172363
2,BBCA,2020-12-30,6770.0,5680.367188
3,BBCA,2021-01-04,6835.0,5734.906738
4,BBCA,2021-01-05,7090.0,5948.864258


## 2. Settings and a small helper

The expected values come from the data validation already done in SQL (queries A to G) and from the research specification: 1,381 return days per instrument, with regimes R1 399, R2 499, R3 393 and Stub 90 days. The tolerances are the ones set in the metric definitions.

In [3]:
FIRST_RETURN_DAY = pd.Timestamp("2021-01-04")
PRIOR_CLOSE_DAY = pd.Timestamp("2020-12-30")
EXPECTED_DAYS = 1381
REGIME_DAYS = {"R1": 399, "R2": 499, "R3": 393, "Stub": 90}
RET_TOL = 1e-12
RF_TOL = 1e-15
CUM_TOL = 1e-9

results = []


def check(name, ok, detail=""):
    results.append(bool(ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  [{detail}]" if detail else ""))

## 3. Structure and alignment

Prices and SQL returns are turned into one table per measure, with dates down the rows and one column per instrument. That layout keeps the four instruments aligned by date, which the CAPM regressions will need.

Python computes returns with `pct_change()` on the full price history (including 2020-12-30), then keeps only the dates from 2021-01-04. That gives the first 2021 return a correct prior close, the same design as the SQL view.

In [4]:
close = db_access.to_wide(prices, "close")
adj = db_access.to_wide(prices, "adj_close")
ret_close_py = close.pct_change().loc[FIRST_RETURN_DAY:]
ret_adj_py = adj.pct_change().loc[FIRST_RETURN_DAY:]

sql_close = db_access.to_wide(sql, "ret_close")
sql_adj = db_access.to_wide(sql, "ret_adj")

check("same tickers in SQL view and prices", list(sql_close.columns) == list(ret_close_py.columns), ", ".join(sql_close.columns))
check("same return dates in SQL view and pandas", sql_close.index.equals(ret_close_py.index), f"{len(sql_close)} vs {len(ret_close_py)}")
check(f"{EXPECTED_DAYS} return days per ticker", (sql_close.notna().sum() == EXPECTED_DAYS).all())
check("no missing returns in pandas", not ret_close_py.isna().any().any())

PASS  same tickers in SQL view and prices  [BBCA, BBRI, BMRI, IHSG]
PASS  same return dates in SQL view and pandas  [1381 vs 1381]
PASS  1381 return days per ticker
PASS  no missing returns in pandas


## 4. M1: simple daily return

Formula: `r_t = P_t / P_(t-1) - 1`. The largest absolute difference between the SQL value and the pandas value, over all dates and instruments, must be below 1e-12. This is done once for the split-adjusted close (the primary basis) and once for the adjusted close (the total-return sensitivity).

In [5]:
d_close = (sql_close - ret_close_py).abs().max().max()
d_adj = (sql_adj - ret_adj_py).abs().max().max()
check("M1 ret_close: SQL equals pandas", d_close < RET_TOL, f"max abs diff {d_close:.2e}")
check("M1 ret_adj: SQL equals pandas", d_adj < RET_TOL, f"max abs diff {d_adj:.2e}")

PASS  M1 ret_close: SQL equals pandas  [max abs diff 2.77e-16]
PASS  M1 ret_adj: SQL equals pandas  [max abs diff 2.77e-16]


## 5. M2: cumulative return

Compounding every daily return, `prod(1 + r) - 1`, must equal the simple price ratio, last close divided by the 2020-12-30 close, minus one. This confirms that the daily returns chain together without a gap. The cumulative figures are price returns, so they ignore dividends and are not adjusted for risk.

In [6]:
cum_from_returns = (1 + ret_close_py).prod() - 1
cum_from_prices = close.loc[close.index.max()] / close.loc[PRIOR_CLOSE_DAY] - 1
d_cum = (cum_from_returns - cum_from_prices).abs().max()
check("M2 compounded returns equal the price ratio", d_cum < CUM_TOL, f"max abs diff {d_cum:.2e}")

pd.DataFrame({"from returns": cum_from_returns, "from prices": cum_from_prices}).round(6)

PASS  M2 compounded returns equal the price ratio  [max abs diff 3.33e-15]


,from returns,from prices
ticker,,
BBCA,-0.102659,-0.102659
BBRI,-0.171688,-0.171688
BMRI,0.267984,0.267984
IHSG,0.015398,0.015398


## 6. M5: risk-free rate and excess return

Daily risk-free rate: `rf = rate_pct / 100 / 252`. Excess return: `e = r - rf`. Python recomputes both from the policy rate and compares with the SQL view. It also checks that the risk-free rate is the same for every instrument on a given date and that no rate is missing.

In [7]:
rf_py = sql["rate_pct"] / 100 / 252
d_rf = (sql["rf_daily"] - rf_py).abs().max()
check("M5 rf_daily equals rate_pct / 100 / 252", d_rf < RF_TOL, f"max abs diff {d_rf:.2e}")
check("M5 no missing risk-free rate", not sql["rf_daily"].isna().any())

d_ex = (sql["excess_close"] - (sql["ret_close"] - rf_py)).abs().max()
check("M5 excess_close equals ret_close minus rf", d_ex < RET_TOL, f"max abs diff {d_ex:.2e}")

rf_wide = db_access.to_wide(sql, "rf_daily")
check("risk-free rate identical across tickers on each date", (rf_wide.nunique(axis=1) == 1).all())

PASS  M5 rf_daily equals rate_pct / 100 / 252  [max abs diff 2.71e-20]
PASS  M5 no missing risk-free rate
PASS  M5 excess_close equals ret_close minus rf  [max abs diff 4.34e-19]
PASS  risk-free rate identical across tickers on each date


## 7. Regime day counts

The regimes are fixed in advance: R1 to 22 Aug 2022, R2 to 17 Sep 2024, R3 to 19 May 2026, then the Stub (descriptive only). The number of return days in each regime must be the same for every instrument. The unequal lengths, especially the 90-day Stub, matter when interpreting later results.

In [8]:
counts = sql.groupby(["ticker", "regime"]).size().unstack()[list(REGIME_DAYS)]
ok = all((counts[r] == n).all() for r, n in REGIME_DAYS.items()) and sum(REGIME_DAYS.values()) == EXPECTED_DAYS
check("regime day counts per ticker", ok, str(REGIME_DAYS))
counts

PASS  regime day counts per ticker  [{'R1': 399, 'R2': 499, 'R3': 393, 'Stub': 90}]


regime,R1,R2,R3,Stub
ticker,,,,
BBCA,399,499,393,90
BBRI,399,499,393,90
BMRI,399,499,393,90
IHSG,399,499,393,90


## 8. Summary

In [9]:
n_fail = results.count(False)
print(f"{len(results) - n_fail} passed, {n_fail} failed")
assert n_fail == 0, "At least one validation check failed. Investigate before using the views."

12 passed, 0 failed
